In [1]:
import pybedtools
from pybedtools import BedTool
import pandas as pd

A = pd.DataFrame({'chrom': ['chr1', 'chr1', 'chr1','chr1'],
                     'start': [1, 1, 5, 20],
                     'end': [10, 10, 7, 30],
					 'name': ['a1', 'a2', 'b', 'c']})

B = pd.DataFrame({'chrom': ['chr1', 'chr1', 'chr1', 'chr1'],
                     'start': [1, 1, 2, 4,],
                     'end': [2, 2, 3, 6,],
					 'name': ['d', 'e', 'f', 'g']})

A_bed = BedTool.from_dataframe(A)
B_bed = BedTool.from_dataframe(B)
print (A)
print (B)


  chrom  start  end name
0  chr1      1   10   a1
1  chr1      1   10   a2
2  chr1      5    7    b
3  chr1     20   30    c
  chrom  start  end name
0  chr1      1    2    d
1  chr1      1    2    e
2  chr1      2    3    f
3  chr1      4    6    g


In [2]:
B_bed.merge().to_dataframe()

,chrom,start,end
0,chr1,1,3
1,chr1,4,6


In [3]:
intersection = A_bed.intersect(B_bed, wb=True)
intersection.to_dataframe()

,chrom,start,end,name,score,strand,thickStart,thickEnd
0,chr1,1,2,a1,chr1,1,2,d
1,chr1,1,2,a1,chr1,1,2,e
2,chr1,2,3,a1,chr1,2,3,f
3,chr1,4,6,a1,chr1,4,6,g
4,chr1,1,2,a2,chr1,1,2,d
5,chr1,1,2,a2,chr1,1,2,e
6,chr1,2,3,a2,chr1,2,3,f
7,chr1,4,6,a2,chr1,4,6,g
8,chr1,5,6,b,chr1,4,6,g


In [4]:
df = intersection.to_dataframe()
agg_result = df.groupby('name').agg(
    thickStart_sum=('thickStart', 'sum'),
    span_sum=('end', lambda x: (x - df.loc[x.index, 'start']).sum())
)
agg_result

,thickStart_sum,span_sum
name,,
a1,13,5
a2,13,5
b,6,1


In [5]:
agg_result.reset_index()

,name,thickStart_sum,span_sum
0,a1,13,5
1,a2,13,5
2,b,6,1


In [ ]:
test1 = BedTool("data/test1.bed")
test2 = BedTool("data/test2.bed")
files = ["data/test1.bed", "data/test2.bed"]
result = test1.cat(*files, postmerge=False).sort().to_dataframe().groupby("name").apply(
	lambda x: BedTool.from_dataframe(x).merge().to_dataframe()
	).reset_index().drop(columns=['level_1'])
result

/tmp/ipykernel_3381637/3069698344.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test1.cat(*files, postmerge=False).sort().to_dataframe().groupby("name").apply(


,name,level_1,chrom,start,end
0,exon,0,chr1,5,15
1,exon,1,chr2,10,20
2,exon,2,chr2,30,40
3,intron,0,chr3,20,30


In [42]:
files[0]

'data/test1.bed'

In [43]:
!cat {files[0]}

chr1	5	10	exon
chr2	10	20	exon

In [45]:
BedTool(files[1]).to_dataframe()

,chrom,start,end,name
0,chr1,5,15,exon
1,chr2,30,40,exon
